<a href="https://colab.research.google.com/github/andi-wahyudi/student-performance-predictor/blob/main/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)
print("Library berhasil diimpor.")

Library berhasil diimpor.


In [ ]:
from google.colab import files
uploaded = files.upload()

needed = {"student-mat-clean-read.csv", "student-por-clean-read.csv"}
missing = needed - set(uploaded.keys())
if missing:
    raise FileNotFoundError("Belum di-upload: " + ", ".join(sorted(missing)))

df_mat = pd.read_csv("student-mat-clean-read.csv")
df_por = pd.read_csv("student-por-clean-read.csv")

print("Matematika:", df_mat.shape)
print("Bahasa Portugis:", df_por.shape)

Saving student-por-clean-read.csv to student-por-clean-read.csv
Saving student-mat-clean-read.csv to student-mat-clean-read.csv
Matematika: (395, 33)
Bahasa Portugis: (649, 33)


In [ ]:
def check_data(df, name):
    print(f"=== {name} ===")
    print("Ukuran:", df.shape)
    print("Kolom G1, G2, G3 tersedia:", all(c in df.columns for c in ["G1", "G2", "G3"]))
    print("Nilai kosong total:", df.isna().sum().sum())
    print("Baris duplikat:", df.duplicated().sum())
    display(df.head(3))

check_data(df_mat, "Matematika")
check_data(df_por, "Bahasa Portugis")

for name, df in [("Matematika", df_mat), ("Bahasa Portugis", df_por)]:
    if not all(c in df.columns for c in ["G1", "G2", "G3"]):
        raise ValueError(f"Kolom G1/G2/G3 tidak lengkap pada {name}.")
    if df["G3"].isna().any():
        raise ValueError(f"Target G3 memiliki nilai kosong pada {name}.")

=== Matematika ===
Ukuran: (395, 33)
Kolom G1, G2, G3 tersedia: True
Nilai kosong total: 0
Baris duplikat: 0


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10


=== Bahasa Portugis ===
Ukuran: (649, 33)
Kolom G1, G2, G3 tersedia: True
Nilai kosong total: 0
Baris duplikat: 0


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,0,yes,no,no,no,yes,yes,yes,no,4,3,2,2,3,3,6,12,13,12


In [ ]:
TARGET = "G3"

def make_xy(df, name):
    X = df.drop(columns=[TARGET]).copy()
    y = df[TARGET].copy()
    print(f"{name}: {X.shape[1]} fitur; G1 in X = {'G1' in X.columns}; G2 in X = {'G2' in X.columns}; G3 hanya target = {TARGET not in X.columns}")
    return X, y

X_mat, y_mat = make_xy(df_mat, "Matematika")
X_por, y_por = make_xy(df_por, "Bahasa Portugis")

display(X_mat.head())
display(y_mat.head())

Matematika: 32 fitur; G1 in X = True; G2 in X = True; G3 hanya target = True
Bahasa Portugis: 32 fitur; G1 in X = True; G2 in X = True; G3 hanya target = True


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10


,G3
0,6
1,6
2,10
3,15
4,10


In [ ]:
def show_feature_types(X, name):
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(exclude=["number"]).columns.tolist()
    print(f"=== {name} ===")
    print("Numerik:", numeric)
    print("Kategorikal:", categorical)
    return numeric, categorical

num_mat, cat_mat = show_feature_types(X_mat, "Matematika")
num_por, cat_por = show_feature_types(X_por, "Bahasa Portugis")

=== Matematika ===
Numerik: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
Kategorikal: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
=== Bahasa Portugis ===
Numerik: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
Kategorikal: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [ ]:
X_train_mat, X_test_mat, y_train_mat, y_test_mat = train_test_split(
    X_mat, y_mat, test_size=0.20, random_state=42
)
X_train_por, X_test_por, y_train_por, y_test_por = train_test_split(
    X_por, y_por, test_size=0.20, random_state=42
)

print("MATEMATIKA:", X_train_mat.shape, X_test_mat.shape, y_train_mat.shape, y_test_mat.shape)
print("PORTUGIS:", X_train_por.shape, X_test_por.shape, y_train_por.shape, y_test_por.shape)

MATEMATIKA: (316, 32) (79, 32) (316,) (79,)
PORTUGIS: (519, 32) (130, 32) (519,) (130,)


In [ ]:
def build_preprocessor(X_train):
    numeric = X_train.select_dtypes(include=["number"]).columns.tolist()
    categorical = X_train.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers = []
    if numeric:
        transformers.append(("numeric", numeric_pipe, numeric))
    if categorical:
        transformers.append(("categorical", categorical_pipe, categorical))

    return ColumnTransformer(transformers=transformers, remainder="drop")

prep_mat = build_preprocessor(X_train_mat)
prep_por = build_preprocessor(X_train_por)
print("Pipeline preprocessing berhasil dibuat.")

Pipeline preprocessing berhasil dibuat.


In [ ]:
X_train_mat_p = prep_mat.fit_transform(X_train_mat)
X_test_mat_p = prep_mat.transform(X_test_mat)

X_train_por_p = prep_por.fit_transform(X_train_por)
X_test_por_p = prep_por.transform(X_test_por)

print("MATEMATIKA - train:", X_train_mat_p.shape, "| test:", X_test_mat_p.shape)
print("PORTUGIS - train:", X_train_por_p.shape, "| test:", X_test_por_p.shape)

MATEMATIKA - train: (316, 58) | test: (79, 58)
PORTUGIS - train: (519, 58) | test: (130, 58)


In [ ]:
def preview_processed(matrix, name):
    dense = matrix.toarray() if hasattr(matrix, "toarray") else matrix
    print(name)
    display(pd.DataFrame(dense).head())

preview_processed(X_train_mat_p, "Hasil preprocessing Matematika")
preview_processed(X_train_por_p, "Hasil preprocessing Bahasa Portugis")

Hasil preprocessing Matematika


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57
0,-0.586396,0.246437,0.423207,-0.625779,-0.056853,-0.456744,0.064432,-1.195981,-0.144396,-0.55443,-0.266496,-0.368473,-0.464408,0.332050,0.626163,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
1,-0.586396,-0.680636,0.423207,0.828236,-1.254553,-0.456744,1.195575,-0.214345,-0.144396,-0.55443,-1.039089,-0.368473,-0.702257,0.643409,0.892831,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
2,-0.586396,-1.607709,0.423207,-0.625779,-0.056853,3.628097,0.064432,-0.214345,1.644982,-0.55443,-1.039089,-0.368473,-0.702257,-0.913385,-0.973844,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,-0.586396,1.173510,0.423207,-0.625779,1.140847,-0.456744,-1.066710,0.767292,0.750293,0.55443,1.278690,0.341512,-0.464408,-0.290667,-0.440508,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
4,-0.586396,-0.680636,-1.434203,0.828236,-1.254553,-0.456744,-1.066710,-0.214345,-1.039085,-0.55443,0.506097,-0.368473,-0.702257,-0.913385,-0.440508,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


Hasil preprocessing Bahasa Portugis


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57
0,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,-0.219949,-1.068345,-0.789616,2.525549,2.272896,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
1,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,2.078442,1.016133,0.065939,0.281097,-0.159337,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
2,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,-0.986079,-0.373519,-0.361839,-0.467054,-0.159337,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
3,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,1.312312,1.016133,-0.789616,0.281097,0.188125,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
4,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,-0.219949,0.321307,-0.147950,-0.467054,-0.159337,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0


In [ ]:
def target_report(y_train, y_test, name):
    print("===", name, "===")
    display(pd.DataFrame({
        "Training": y_train.describe(),
        "Testing": y_test.describe()
    }).round(2))

target_report(y_train_mat, y_test_mat, "Matematika")
target_report(y_train_por, y_test_por, "Bahasa Portugis")

=== Matematika ===


,Training,Testing
count,316.00,79.00
mean,10.33,10.77
std,4.59,4.56
min,0.00,0.00
25%,8.00,8.00
50%,11.00,11.00
75%,13.00,14.50
max,20.00,19.00


=== Bahasa Portugis ===


,Training,Testing
count,519.00,130.00
mean,11.79,12.35
std,3.25,3.13
min,0.00,0.00
25%,10.00,11.00
50%,12.00,12.00
75%,14.00,14.00
max,19.00,19.00


In [ ]:
OUT = Path("/content/preprocessing_outputs")
OUT.mkdir(exist_ok=True)

def save_outputs(X_train, X_test, y_train, y_test, prefix, preprocessor):
    train_df = X_train.copy()
    train_df[TARGET] = y_train
    test_df = X_test.copy()
    test_df[TARGET] = y_test

    train_df.to_csv(OUT / f"{prefix}_train.csv", index=False)
    test_df.to_csv(OUT / f"{prefix}_test.csv", index=False)
    joblib.dump(preprocessor, OUT / f"{prefix}_preprocessor.joblib")

save_outputs(X_train_mat, X_test_mat, y_train_mat, y_test_mat, "matematika", prep_mat)
save_outputs(X_train_por, X_test_por, y_train_por, y_test_por, "portugis", prep_por)

print("File hasil:")
for p in sorted(OUT.iterdir()):
    print(p.name)

File hasil:
matematika_preprocessor.joblib
matematika_test.csv
matematika_train.csv
portugis_preprocessor.joblib
portugis_test.csv
portugis_train.csv


In [ ]:
zip_path = Path("/content/preprocessing_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        z.write(p, arcname=p.name)

print("ZIP siap:", zip_path)
from google.colab import files
files.download(str(zip_path))

ZIP siap: /content/preprocessing_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>